# Problem 1 (2.5 pts) — Binding C++ and Eigen with `pybind11`

Python is convenient for high-level orchestration, but tight numerical loops over arrays or custom data structures incur massive interpreter overhead. In this problem, you will expose C++ and [Eigen](https://eigen.tuxfamily.org/) routines to Python using [`pybind11`](https://pybind11.readthedocs.io/en/stable/).

### Task 1 (1.5 pts) — Fixing and Exposing the Starter Project

Clone the starter repository with submodules:
```bash
git clone --recursive https://github.com/Aelphy/pybind_seminar
```
Make sure that both `external/eigen` and `external/pybind11` are non-empty (`git submodule update --init --recursive` if needed).

Follow these steps to debug, build, and benchmark the C++ extension module:
1. Create a `build` directory and run `cmake` and `make` as described in `README.md`. Read the first compiler error carefully.
2. Inspect `CMakeLists.txt`, `include/my_file.hpp`, `src/my_file.cpp`, and `src/pybind_connector.cpp`. Fix the broken build step by step:
   - Make sure the module name in `PYBIND11_MODULE(..., m)` matches the CMake target name (`my_project`).
   - Make sure `src/my_file.cpp` is compiled into the target in `CMakeLists.txt`.
   - Add missing standard library headers (`<vector>`, `<stdexcept>`) and forward declarations in `include/my_file.hpp`.
   - Expose `print_arg` (all three overloads), `factorial`, `binomial_coefficient`, `double_factorial`, and `dot_product` in `src/pybind_connector.cpp`. To disambiguate overloaded C++ functions with the same name, use `static_cast<Ret (*)(Args...)>(&func)` or `py::overload_cast<Args...>(&func)` (see the [pybind11 documentation on overloaded functions](https://pybind11.readthedocs.io/en/stable/classes.html#overloaded-methods)).
3. Once those functions compile and can be imported into Python (by adding your `build/` directory to `sys.path`), fix the return type and body of `matmul` in `src/my_file.cpp` so that it multiplies two row-major Eigen matrices (`Eigen::Matrix<double, Eigen::Dynamic, Eigen::Dynamic, Eigen::RowMajor>`) and returns the product.
4. Benchmark your C++ `my_project.matmul(A, B)` against `np.dot(A, B)` (and a naive Python or non-vectorized baseline) for square matrices of sizes $N \in \{64, 128, 256, 512, 1024, 2048\}$ on a log-log runtime plot. Why is `np.dot` still competitive with or faster than a single-threaded Eigen multiplication for large $N$? What compiler flags (`-O3 -march=native`) affect Eigen's performance?

In [ ]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

# Adjust the path below to point to your compiled pybind_seminar/build directory
# sys.path.insert(0, "/path/to/pybind_seminar/build")
# import my_project

# TODO: Test print_arg, factorial, binomial_coefficient, double_factorial, dot_product
# and benchmark my_project.matmul vs np.dot across matrix sizes.


### Task 2 (1 pt) — Zero-Copy 2D Stencil Sweep in C++ (`Eigen::Ref`)

When binding `Eigen::Matrix` by value in `matmul`, `pybind11` copies the NumPy array buffer into a new `Eigen::Matrix` on every call. For iterative PDE solvers (which we study in Problem 3), copying the grid on every step destroys performance.

1. Add a new C++ function to `src/my_file.cpp` and expose it in `src/pybind_connector.cpp`:
   ```cpp
   using RowMatrixXd = Eigen::Matrix<double, Eigen::Dynamic, Eigen::Dynamic, Eigen::RowMajor>;
   void jacobi_sweep_2d(Eigen::Ref<RowMatrixXd> u, const Eigen::Ref<const RowMatrixXd>& f, double h2, int n_steps);
   ```
   Using `Eigen::Ref<RowMatrixXd>` allows `pybind11` to mutate a contiguous row-major (`dtype=np.float64`, C-order) NumPy array **in place without copying memory**.
2. Inside `jacobi_sweep_2d`, perform `n_steps` iterations of the 2D 5-point Poisson Jacobi relaxation on the interior points $1 \le i < R-1, \; 1 \le j < C-1$:
   $$u^{(k+1)}_{i,j} = \frac{1}{4}\left(u^{(k)}_{i-1,j} + u^{(k)}_{i+1,j} + u^{(k)}_{i,j-1} + u^{(k)}_{i,j+1} + h^2 f_{i,j}\right)$$
   *(Keep boundary rows/columns $i \in \{0, R-1\}$ and $j \in \{0, C-1\}$ fixed).*
3. Verify that calling `my_project.jacobi_sweep_2d(u, f, h**2, 100)` modifies `u` in place and matches 100 steps of vectorized NumPy slicing to machine precision ($\|u_{\text{cpp}} - u_{\text{numpy}}\|_\infty < 10^{-14}$). Compare the runtime for a $500 \times 500$ grid against:
   - Pure Python nested `for` loops (run for 1–5 steps and extrapolate),
   - Vectorized NumPy slicing (`u[1:-1, 1:-1] = 0.25 * (...)`),
   - Your C++ `jacobi_sweep_2d`.
   Why is the C++ implementation faster than vectorized NumPy even though NumPy also calls compiled C loops? *(Hint: count how many temporary $500 \times 500$ arrays NumPy allocates per step vs. cache locality in C++).*

In [ ]:
# TODO: Verify zero-copy in-place behavior and benchmark jacobi_sweep_2d vs NumPy slicing


*please don't forget to include your report (and your modified `src/my_file.cpp` and `src/pybind_connector.cpp` snippets)*

# Problem 2 (3.5 pts) — Opinion Crystallization: Minority Takeover & Buying Celebrities

Last week, we studied the **Friedkin-Johnsen (FJ) model** with epoch-by-epoch *opinion crystallization*:
$$(I - \Lambda W) x^{(k)} = (I - \Lambda) s^{(k)}, \qquad s^{(k+1)} = x^{(k)},$$
where $W \in \mathbb{R}^{N \times N}$ is row-stochastic, $\Lambda = \operatorname{diag}(\lambda_1, \dots, \lambda_N)$ contains each agent's susceptibility to peer pressure, and $s^{(k)}$ is the baseline conviction at epoch $k$.

In Week 4, both political clusters had equal size ($N/2$ vs. $N/2$) and a single sticky leader. In real social networks, groups are rarely balanced, and influence campaigns must decide **how to allocate a limited budget across multiple influencers**.

---

### Task 1 (1.5 pts) — Imbalanced Groups & The Vocal Minority Phase Transition

Consider a population of $N = 400$ agents split into two imbalanced communities:
- **Cluster A (Vocal Minority)**: size $N_A = \lfloor f N \rfloor$ with fraction $f = 0.15$ ($15\%$ of the population), initial convictions $s_i^{(0)} \sim \mathcal{U}(0.4, 0.9)$ (mean $\bar{s}_A \approx +0.65$).
- **Cluster B (Silent Majority)**: size $N_B = N - N_A$ ($85\%$ of the population), initial convictions $s_i^{(0)} \sim \mathcal{U}(-0.9, -0.4)$ (mean $\bar{s}_B \approx -0.65$).

Suppose every agent in Cluster A has susceptibility $\lambda_A \in (0, 1)$, and every agent in Cluster B has susceptibility $\lambda_B \in (0, 1)$. After row-normalization of $W$, each member of Cluster A places total weight $1 - p_{AB}$ on peers inside A and $p_{AB}$ on peers in B; symmetrically, each member of Cluster B places weight $1 - p_{BA}$ on peers inside B and $p_{BA}$ on peers in A.

1. **Mean-Field Derivation**: By averaging the single-epoch FJ equilibrium $(I - \Lambda W)x = (I - \Lambda)s$ over Cluster A and Cluster B (assuming homogeneous intra-/inter-cluster weights), write the $2 \times 2$ system for the cluster means $(\bar{x}_A, \bar{x}_B)$ in terms of $(\bar{s}_A, \bar{s}_B)$, $(\lambda_A, \lambda_B)$, and $(p_{AB}, p_{BA})$.
   - In the limit of repeated crystallization epochs $k \to \infty$ (without resetting any single leader's conviction), show that the entire population converges to a global consensus $x_\infty \mathbf{1}$ if the interaction graph is strongly connected, and derive the exact condition under which the **$15\%$ minority flips the $85\%$ majority** ($x_\infty > 0$).
   - How does this condition depend on the **stubbornness ratio** $\frac{1 - \lambda_A}{1 - \lambda_B}$ and the **cross-group attention ratio** $\frac{p_{BA}}{p_{AB}}$ (e.g., algorithmic amplification of the minority's posts into the majority's feed)?
2. **Numerical Phase Diagram**: Using the starter code below, sweep the minority susceptibility $\lambda_A \in [0.05, 0.85]$ and algorithmic amplification factor $\gamma_{\text{amp}} \in [0.5, 15.0]$ (which scales the raw bridge weights from B to A), compute the population mean opinion $\bar{x}^{(T)}$ after $T = 40$ crystallization epochs, and overlay your theoretical boundary curve $\bar{x}_\infty = 0$ on the 2D heatmap.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def build_imbalanced_network(n=400, f_minority=0.15, gamma_amp=1.0, seed=42):
    """
    Constructs a row-stochastic interaction matrix W for an imbalanced two-group network.
    gamma_amp amplifies the attention that the majority (Cluster B) pays to the minority (Cluster A).
    """
    rng = np.random.default_rng(seed)
    n_A = int(round(f_minority * n))
    n_B = n - n_A

    s_0 = np.zeros((n, 1))
    s_0[:n_A, 0] = rng.uniform(0.4, 0.9, size=n_A)
    s_0[n_A:, 0] = rng.uniform(-0.9, -0.4, size=n_B)

    raw_W = np.zeros((n, n))
    # Intra-cluster ties (normalized by cluster size so per-group total attention is comparable)
    raw_W[:n_A, :n_A] = rng.uniform(0.2, 0.6, size=(n_A, n_A)) / n_A
    raw_W[n_A:, n_A:] = rng.uniform(0.2, 0.6, size=(n_B, n_B)) / n_B

    # Cross-cluster bridge ties
    raw_W[:n_A, n_A:] = rng.uniform(0.01, 0.05, size=(n_A, n_B)) / n_B
    raw_W[n_A:, :n_A] = gamma_amp * rng.uniform(0.01, 0.05, size=(n_B, n_A)) / n_A

    np.fill_diagonal(raw_W, 0.0)
    W = raw_W / raw_W.sum(axis=1, keepdims=True)
    return W, s_0, n_A, n_B

# TODO:
# 1. Simulate crystallization across a grid of (lambda_A, gamma_amp) with fixed lambda_B = 0.65
#    (Hint: notice that if Lambda and W are fixed across the T_epochs, you do NOT need to invert
#     (I - Lambda W) T_epochs times!)
# 2. Plot the 2D phase diagram of the final population mean opinion and overlay the analytical boundary.


---

### Task 2 (2 pts) — Buying $k$ Celebrities: Echo-Chamber Cannibalization & Rank-$k$ Woodbury

Now suppose a campaign has a budget to "buy" exactly $k = 5$ influencers out of a candidate pool of $M = 40$ prominent nodes in a network of $N = 1000$ agents.

When a set of influencers $S = \{j_1, \dots, j_k\}$ is bought by the positive campaign:
- Their innate convictions are pinned to $s_j = +1.0$ for all $j \in S$,
- Their susceptibilities are reduced to $\lambda_j = \lambda_{\text{bought}} = 0.01$ (they become near-stubborn broadcasters of $+1.0$).

All other agents $i \notin S$ keep their baseline convictions $s_i^{(0)}$ and baseline susceptibilities $\lambda_i^{(0)}$. We want to choose $S$ ($|S| = k$) to maximize the **average steady-state population opinion**
$$J(S) = \frac{1}{N} \mathbf{1}^T x(S), \qquad \text{where} \quad \big(I - \Lambda(S) W\big) x(S) = \big(I - \Lambda(S)\big) s(S).$$

#### Why Brute-Force Fails and Sherman-Morrison-Woodbury Shines
Evaluating a single subset $S$ by solving $(I - \Lambda(S) W) x = b$ from scratch costs $O(N^3)$. In greedy subset selection (testing $M$ candidates in $k$ rounds, or evaluating thousands of combinations), doing an $O(N^3)$ solve per candidate is prohibitively slow.

Observe that changing $\lambda_j^{(0)} \to \lambda_{\text{bought}}$ for $j \in S = \{j_1, \dots, j_k\}$ only modifies **$k$ rows** of the base matrix $A_0 = I - \Lambda_0 W$:
$$A(S) = A_0 + U V^T,$$
where $U \in \mathbb{R}^{N \times k}$ consists of standard basis columns $e_{j_1}, \dots, e_{j_k}$, and $V^T \in \mathbb{R}^{k \times N}$ contains the $k$ row perturbations $(\lambda_j^{(0)} - \lambda_{\text{bought}}) W_{j, :}$.

By the **[Sherman-Morrison-Woodbury formula](https://en.wikipedia.org/wiki/Woodbury_matrix_identity)**:
$$\big(A_0 + U V^T\big)^{-1} b = A_0^{-1} b - A_0^{-1} U \left(I_k + V^T A_0^{-1} U\right)^{-1} V^T \big(A_0^{-1} b\big).$$
Once $A_0^{-1}$ is precomputed **once** in $O(N^3)$, notice that:
- $A_0^{-1} U$ simply extracts $k$ columns of $A_0^{-1}$ ($O(Nk)$ memory lookup, zero flops!),
- $I_k + V^T A_0^{-1} U$ is a tiny $k \times k$ matrix ($5 \times 5$!),
- Even better: since we only need the scalar objective $J(S) = \frac{1}{N} \mathbf{1}^T x(S)$, if we precompute the **influence sensitivity vector** $w_0^T = \frac{1}{N} \mathbf{1}^T A_0^{-1}$, we can evaluate $J(S)$ for **any** subset $S$ of size $k$ in $O(k^2 N + k^3)$ time!

#### Your Tasks:
1. **Derive and implement the fast Woodbury evaluator** `evaluate_subset_woodbury(S, A0_inv, W, lambdas_0, s_0, lambda_bought=0.01)` that computes $x(S)$ and $J(S)$ using the rank-$k$ Woodbury update without forming or inverting any $N \times N$ matrix. Verify on 10 random subsets of size $k=5$ that it matches `np.linalg.solve(A_S, b_S)` to within $10^{-12}$.
2. **Compare 3 Influencer Selection Strategies** for budgets $k = 1, 2, \dots, 10$ on the network generated below (which has three archetypes of candidate influencers: **Type I** mega-celebrities inside Cluster A, **Type II** overlapping-audience celebrities in Cluster B, and **Type III** diverse bridge influencers in Cluster B):
   - **Naive Follower-Count Heuristic**: Pick the $k$ candidates with the largest total follower attention $\sum_{i=1}^N W_{i, j}$.
   - **Independent Single-Node Marginal Gain**: Score each candidate $j$ individually by $J(\{j\}) - J(\emptyset)$ and pick the top $k$.
   - **Sequential Greedy with Rank-$k$ Woodbury**: In each round $r = 1, \dots, k$, pick $j^* = \arg\max_{j \notin S} J(S \cup \{j\})$ using your Woodbury evaluator.
3. **Measure Speedup & Explain the Sociology**:
   - Report the wall-clock speedup of greedy selection using your Woodbury evaluator vs. brute-force `np.linalg.solve`.
   - Plot $J(S)$ vs. budget $k \in \{1, \dots, 10\}$ for all three strategies, and inspect which influencer types (I, II, or III) each strategy buys. Why does buying the highest-follower celebrities (Type I) fail, and why does Independent Marginal Gain underperform Sequential Greedy when two celebrities share the same audience (Type II **echo-chamber cannibalization**)?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

def generate_celebrity_network(n=1000, seed=101):
    """
    Generates a polarized network of N=1000 agents with M=30 candidate celebrities
    divided into three distinct structural archetypes:
      - Candidates 0..9   (Type I):  Mega-celebrities inside Cluster A (huge total follower weight,
                                     but watched almost exclusively by already-positive Cluster A).
      - Candidates 10..19 (Type II): Tightly coupled mutual-follower clique in sub-community B1
                                     (160 swing voters in Cluster B all follow these 10 co-influencers,
                                     who also heavily influence each other).
      - Candidates 20..29 (Type III): Diverse regional/niche bridge influencers in Cluster B (each
                                      reaches a disjoint sub-community of 35 agents in Cluster B).
    """
    rng = np.random.default_rng(seed)
    n_A = 380   # Positive minority (38%)
    n_B = 620   # Negative majority (62%)

    s_0 = np.zeros(n)
    s_0[:n_A] = rng.uniform(0.5, 0.9, size=n_A)
    s_0[n_A:] = rng.uniform(-0.8, -0.5, size=n_B)

    lambdas_0 = rng.uniform(0.65, 0.85, size=n)

    raw_W = np.zeros((n, n))
    raw_W[:n_A, :n_A] = rng.uniform(0.5, 1.5, size=(n_A, n_A)) / n_A
    raw_W[n_A:, n_A:] = rng.uniform(0.5, 1.5, size=(n_B, n_B)) / n_B
    raw_W[:n_A, n_A:] = rng.uniform(0.02, 0.08, size=(n_A, n_B)) / n_B
    raw_W[n_A:, :n_A] = rng.uniform(0.02, 0.08, size=(n_B, n_A)) / n_A

    candidates = []
    candidate_types = []

    # Type I: 10 mega-celebrities inside Cluster A (indices 0..9)
    for j in range(10):
        raw_W[:n_A, j] += rng.uniform(12.0, 15.0, size=n_A)
        candidates.append(j)
        candidate_types.append("Type I (Cluster A Mega-Star)")

    # Type II: 10 coupled celebrities in sub-community B1 (160 agents: n_A .. n_A+160)
    B1 = slice(n_A, n_A + 160)
    B1_hubs = slice(n_A, n_A + 10)
    lambdas_0[B1] = 0.90
    lambdas_0[B1_hubs] = 0.98
    raw_W[B1_hubs, :] = 0.0
    raw_W[B1_hubs, B1_hubs] = rng.uniform(5.0, 6.0, size=(10, 10))
    for idx in range(10):
        j = n_A + idx
        raw_W[B1, j] += rng.uniform(1.5, 2.0, size=160)
        candidates.append(j)
        candidate_types.append("Type II (Coupled B1 Clique)")

    # Type III: 10 disjoint niche influencers in Cluster B (each reaching 35 agents)
    for idx in range(10):
        sub = slice(n_A + 160 + idx * 35, n_A + 160 + (idx + 1) * 35)
        lambdas_0[sub] = 0.90
        j = n_A + 160 + idx * 35
        raw_W[sub, j] += rng.uniform(3.0, 4.0, size=35)
        candidates.append(j)
        candidate_types.append("Type III (Disjoint Niche Leader)")

    np.fill_diagonal(raw_W, 0.0)
    W = raw_W / raw_W.sum(axis=1, keepdims=True)

    return W, s_0, lambdas_0, np.array(candidates), candidate_types

W, s_0, lambdas_0, candidates, candidate_types = generate_celebrity_network()

# TODO:
# 1. Precompute A0_inv = np.linalg.inv(np.eye(len(s_0)) - np.diag(lambdas_0) @ W)
# 2. Implement evaluate_subset_woodbury(S, ...) and verify against np.linalg.solve
# 3. Compare Naive Follower-Count, Independent Marginal Gain, and Sequential Greedy for k = 1..10


*please don't forget to include your report*

# Problem 3 (4 pts) — Revisiting Week 2's CPU Die at Micron Scale: Sparse LU Fill-In & The 256 GB Cluster Wall

In Week 2 (Problem 5), you modeled steady-state heat conduction on a $20\text{ mm} \times 20\text{ mm}$ silicon CPU die ($L = 0.02\text{ m}$, $k_{\text{Si}} = 150\text{ W/(m}\cdot\text{K)}$, heatsink boundary $T_{\text{amb}} = 25\,^\circ\text{C}$, throttle threshold $T_{\text{throttle}} = 95\,^\circ\text{C}$) with three functional blocks (**Core A**, **Core B**, and the **Memory Controller**), and found the largest grid $N$ your laptop could handle.

Real modern processors are heterogeneous: to prevent heat from **Core A** from leaking into the temperature-sensitive **Memory Controller**, chip designers surround Core A with a micron-scale **silicon-dioxide thermal isolation trench** ($k_{\text{ox}} = 0.05\text{ W/(m}\cdot\text{K)}$) while draining Core A's heat toward the left heatsink through a narrow **copper micro-heatpipe** ($k_{\text{Cu}} = 400\text{ W/(m}\cdot\text{K)}$). The governing PDE is the variable-conductivity steady heat equation:
$$-\nabla \cdot \big(k(x, y) \nabla T(x, y)\big) = q(x, y) \quad \text{in } \Omega = (0, L)^2, \qquad T = T_{\text{amb}} = 25\,^\circ\text{C} \quad \text{on } \partial\Omega.$$

We discretize $\Omega$ on an $n \times n$ cell-centered finite-volume grid ($h = L / n$, total unknowns $N = n^2$) using the **harmonic average** of $k$ across adjacent cell faces:
$$k_{i+1/2, j} = \frac{2 k_{i,j} k_{i+1,j}}{k_{i,j} + k_{i+1,j}}, \qquad k_{i, j+1/2} = \frac{2 k_{i,j} k_{i,j+1}}{k_{i,j} + k_{i,j+1}},$$
which yields a **symmetric positive definite (SPD)** 5-point sparse linear system $A x = b$ of size $N \times N$.

---

### Task 1 (2 pts) — Why Coarse Grids + Interpolation Fail (Plus Sparse LU Fill-In Scaling)

When faced with a fine grid, a tempting shortcut is to solve the PDE on a coarse grid ($n_{\text{coarse}}$, say $8\times$ smaller per axis) on a laptop and interpolate $T(x, y)$ to the fine grid.

1. **Vectorized Sparse Assembly**: Complete `assemble_cpu_die(k, q, L=0.02, T_amb=25.0)` below to construct the $N \times N$ sparse matrix $A$ in `scipy.sparse.csc_matrix` format and RHS vector $b$ **without any Python `for` loops**.
2. **Defeating the Coarse-Grid + Interpolation Shortcut**:
   - Using the default trench/via widths `(w_trench=8e-5, w_via=1e-4)`, solve the system on the resolved grid $n_{\text{fine}} = 400$ and report the peak die temperature $\max T(x,y)$. Does the CPU exceed the thermal throttle threshold $T_{\text{throttle}} = 95\,^\circ\text{C}$?
   - Now test **all three coarse-grid shortcuts** at $n_{\text{coarse}} = 50$ ($8 \times 8$ fine cells per coarse cell):
     1. **Point-Sampling**: Evaluate `cpu_die_fields(50)` directly on the $50 \times 50$ grid.
     2. **Arithmetic Block-Averaging**: Reshape `k_fine` into `(50, 8, 50, 8)` and average `k_arith = k_blk.mean(axis=(1, 3))`.
     3. **Harmonic Block-Averaging**: Average `k_harm = 1.0 / (1.0 / k_blk).mean(axis=(1, 3))`.
   - Plot the 2D temperature fields side by side and compare their predicted peak temperatures against the $95\,^\circ\text{C}$ throttle threshold (and silicon's $1414\,^\circ\text{C}$ melting point!). Explain mathematically why **arithmetic averaging erases the thin oxide trench** while **harmonic averaging seals the narrow copper heatpipe shut**.
3. **The Three Memory Scaling Laws**: For $n \in \{64, 128, 256, 512\}$, measure and plot on a log-log scale (bytes vs. $n$):
   - Theoretical dense matrix storage $8 N^2 = 8 n^4$ bytes (which killed your laptop in Week 2 around $n \approx 150\text{–}200$),
   - Sparse CSC storage `sparse_nbytes(A)`,
   - Direct sparse LU factor storage `splu_nbytes(lu)` for `splu(A, permc_spec='NATURAL')` (run on $n \in \{64, 128, 256\}$) and `splu(A, permc_spec='COLAMD')` (run on $n \in \{64, 128, 256, 512\}$).
   - Fit the power law $\text{Memory}_{\text{LU}}(n) \propto n^\beta$ for both orderings and predict the grid size $n$ at which each exceeds **16 GB** (laptop crash) vs. **256 GB** (cluster limit).

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
import time
import resource

def cpu_die_fields(n, w_trench=8e-5, w_via=1e-4):
    """
    Upgrades Week 2 Problem 5 (20 mm x 20 mm CPU die, k_Si = 150 W/(m*K), T_amb = 25 °C)
    with a sub-millimeter silicon-dioxide thermal isolation trench (k_ox = 0.05 W/(m*K))
    around Core A and a narrow copper micro-heatpipe (k_Cu = 400 W/(m*K)) draining Core A.
    """
    L = 0.02  # 20 mm die
    h = L / n
    x = (np.arange(n) + 0.5) * h
    X, Y = np.meshgrid(x, x, indexing="ij")

    k = np.full((n, n), 150.0)

    # Circular oxide isolation trench around Core A (center = (6 mm, 6 mm), radius = 2.2 mm)
    dx = X - 0.006
    dy = Y - 0.006
    r_A = np.hypot(dx, dy)
    trench = np.abs(r_A - 0.0022) <= w_trench
    # Narrow copper micro-heatpipe draining Core A across the left side of the trench
    copper_via = (np.abs(dy) <= w_via) & (dx > -0.0032) & (dx < -0.0012)
    k[trench] = 0.05
    k[copper_via] = 400.0

    q = np.zeros((n, n))
    # Core A (r <= 1.8 mm), Core B ([12, 16]x[4, 8] mm), Memory Controller ([6, 14]x[14, 18] mm)
    q[r_A <= 0.0018] = 6.5e8
    q[(X >= 0.012) & (X <= 0.016) & (Y >= 0.004) & (Y <= 0.008)] = 5.0e8
    q[(X >= 0.006) & (X <= 0.014) & (Y >= 0.014) & (Y <= 0.018)] = 2.0e8

    return X, Y, k, q

def sparse_nbytes(M):
    """Returns exact storage in bytes of a CSR/CSC sparse matrix."""
    return M.data.nbytes + M.indices.nbytes + M.indptr.nbytes

def splu_nbytes(lu):
    """Returns exact storage in bytes of the L and U factors in a SuperLU object."""
    return sparse_nbytes(lu.L) + sparse_nbytes(lu.U)

def assemble_cpu_die(k, q, L=0.02, T_amb=25.0):
    """
    Assembles the N x N (N = n^2) symmetric positive definite sparse matrix A (in CSC format)
    and RHS vector b for -div(k grad T) = q on (0, L)^2 with Dirichlet T = T_amb on boundary.
    """
    # TODO: Implement loop-free sparse assembly of A and RHS b using harmonic face averages of k
    raise NotImplementedError("Implement vectorized 5-point finite-volume assembly")


---

### Task 2 (2 pts) — Crossing the Laptop Death Zone on the 256 GB Cluster

Use Euler cluster [documentation](https://docs.hpc.ethz.ch/tutorials/getting-started/#setting-up-your-environment) for loading modules and submitting jobs.

When chip features shrink to micron scale (`w_trench=1.2e-5, w_via=1.6e-5`, i.e. $24\,\mu\text{m}$ trench and $32\,\mu\text{m}$ copper via), resolving them requires a grid of $n = 3500$ ($N = 12{,}250{,}000$ unknowns, $h = 5.7\,\mu\text{m}$) to $n = 5000$ ($N = 25{,}000{,}000$ unknowns, $h = 4.0\,\mu\text{m}$).

1. **Direct Sparse Solver on the Cluster (`n = 3500`)**:
   - On the cluster node (256 GB RAM), assemble the micron-scale system `cpu_die_fields(3500, w_trench=1.2e-5, w_via=1.6e-5)` and solve it using `splu(A, permc_spec='COLAMD')`.
   - Record the peak resident memory (`resource.getrusage(resource.RUSAGE_SELF).ru_maxrss`), the exact factor memory `splu_nbytes(lu)`, wall-clock time, and the peak temperatures inside **Core A**, **Core B**, and the **Memory Controller**.
   - Explain why this factorization crashes with `MemoryError` / OS OOM-kill on a 16 GB laptop even though the sparse matrix $A$ itself takes less than **0.75 GB**.
2. **Beating the Fill-In Wall — Preconditioned Conjugate Gradient (`n = 3500` and `n = 5000`)**:
   - Because $A$ is symmetric positive definite, we can use **Conjugate Gradient** (`scipy.sparse.linalg.cg`)—just as in Week 2!—equipped with a **Jacobi (diagonal) preconditioner** $M^{-1} = \operatorname{diag}(A)^{-1}$ (passed via `M=sp.diags(1.0 / A.diagonal())` or a `LinearOperator`).
   - First, compare the number of CG iterations required at $n = 200$ **with** vs. **without** the diagonal preconditioner (`rtol=1e-8`). Why does the $8000\times$ conductivity contrast between $k_{\text{ox}} = 0.05$ and $k_{\text{Cu}} = 400$ cause unpreconditioned CG to stall, and how does diagonal rescaling fix it?
   - Next, solve the micron-scale $n = 3500$ ($N = 1.225 \times 10^7$) and $n = 5000$ ($N = 2.5 \times 10^7$) systems using preconditioned CG (`rtol=1e-8`), and compare peak RAM, runtime, and peak core temperatures against `splu`.

In [ ]:
# TODO:
# 1. Compare coarse-grid shortcuts (point-sampling, arithmetic avg, harmonic avg at n=50) vs resolved grid (n=400)
# 2. Measure CSC vs LU(NATURAL) vs LU(COLAMD) memory scaling for n in {64, 128, 256, 512}
# 3. On the cluster, solve n = 3500 with splu(COLAMD) vs Preconditioned CG, and push Preconditioned CG to n = 5000


*please don't forget to include your report*